# Eventalign Pipeline — End-to-End Test

This notebook runs the full baleen eventalign pipeline on real test data:

1. **Input validation** — check files exist, BAMs are indexed
2. **BAM stats & contig filtering** — depth-based filtering
3. **f5c eventalign** — signal-level alignment
4. **Signal extraction & DTW** — per-position pairwise distance matrices
5. **Visualization** — heatmaps of DTW distance matrices

### Expected directory structure

```
testdata/
├── control_0/
│   ├── blow5/nanopore.blow5
│   └── fastq/pass.fq.gz
├── control_0.bam
├── control_0.bam.bai
├── control_0.bam.csi
├── native_0/
│   ├── blow5/nanopore.blow5
│   └── fastq/pass.fq.gz
├── native_0.bam
├── native_0.bam.bai
├── native_0.bam.csi
├── ref.fa
├── ref.fa.dict
└── ref.fa.fai
```

## 0. Setup & Path Configuration

In [ ]:
import logging
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Enable INFO-level logging so pipeline progress is visible
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)

# Adjust this path to point to your testdata directory
TESTDATA = Path("../testdata").resolve()
assert TESTDATA.exists(), f"testdata directory not found: {TESTDATA}"

# ---------- File paths ----------
# Native sample
NATIVE_BAM   = TESTDATA / "native_0.bam"
NATIVE_FASTQ = TESTDATA / "native_0" / "fastq" / "pass.fq.gz"
NATIVE_BLOW5 = TESTDATA / "native_0" / "blow5" / "nanopore.blow5"

# Control (IVT) sample
IVT_BAM   = TESTDATA / "control_0.bam"
IVT_FASTQ = TESTDATA / "control_0" / "fastq" / "pass.fq.gz"
IVT_BLOW5 = TESTDATA / "control_0" / "blow5" / "nanopore.blow5"

# Reference
REF_FASTA = TESTDATA / "ref.fa"

# Verify all files exist
for label, path in [
    ("native BAM",   NATIVE_BAM),
    ("native FASTQ", NATIVE_FASTQ),
    ("native BLOW5", NATIVE_BLOW5),
    ("IVT BAM",      IVT_BAM),
    ("IVT FASTQ",    IVT_FASTQ),
    ("IVT BLOW5",    IVT_BLOW5),
    ("ref FASTA",    REF_FASTA),
]:
    status = "\u2705" if path.exists() else "\u274c MISSING"
    print(f"  {status}  {label:15s}  {path}")

print(f"\nAll paths verified from: {TESTDATA}")

## 1. BAM Statistics & Contig Filtering

Before running the full pipeline, inspect per-contig mapping statistics and
see which contigs pass depth filtering.

In [ ]:
from baleen.eventalign._bam import get_contig_stats, filter_contigs

native_stats = get_contig_stats(NATIVE_BAM)
ivt_stats = get_contig_stats(IVT_BAM)

print(f"Native BAM: {len(native_stats)} contigs with mapped reads")
print(f"IVT BAM:    {len(ivt_stats)} contigs with mapped reads")
print()

# Show top contigs by depth
print("--- Native (top 20 by mean depth) ---")
for s in sorted(native_stats.values(), key=lambda x: x.mean_depth, reverse=True)[:20]:
    print(f"  {s.contig:40s}  reads={s.mapped_reads:5d}  depth={s.mean_depth:.1f}")

print()
print("--- IVT (top 20 by mean depth) ---")
for s in sorted(ivt_stats.values(), key=lambda x: x.mean_depth, reverse=True)[:20]:
    print(f"  {s.contig:40s}  reads={s.mapped_reads:5d}  depth={s.mean_depth:.1f}")

In [ ]:
MIN_DEPTH = 15

passed_contigs, filter_results = filter_contigs(
    native_stats, ivt_stats, min_depth=MIN_DEPTH
)

print(f"Contigs passing filter (min_depth={MIN_DEPTH}): {len(passed_contigs)}")
for c in passed_contigs:
    nd = native_stats[c].mean_depth
    id_ = ivt_stats[c].mean_depth
    print(f"  {c:40s}  native_depth={nd:.1f}  ivt_depth={id_:.1f}")

print(f"\nContigs filtered out: {len(filter_results) - len(passed_contigs)}")
for fr in filter_results:
    if not fr.passed:
        print(f"  {fr.contig:40s}  reason={fr.reason.value}")

## 2. Run Full Pipeline

This calls `run_pipeline()` which orchestrates:
- f5c indexing
- BAM validation & contig filtering  
- Per-contig BAM splitting
- f5c eventalign
- Signal extraction & pairwise DTW computation

Results are saved to `output/` as a pickle file.

In [ ]:
from baleen.eventalign import run_pipeline, save_results, load_results

OUTPUT_DIR = Path("../output/eventalign_test").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results, metadata = run_pipeline(
    native_bam=NATIVE_BAM,
    native_fastq=NATIVE_FASTQ,
    native_blow5=NATIVE_BLOW5,
    ivt_bam=IVT_BAM,
    ivt_fastq=IVT_FASTQ,
    ivt_blow5=IVT_BLOW5,
    ref_fasta=REF_FASTA,
    min_depth=MIN_DEPTH,
    use_cuda=None,         # auto-detect GPU
    use_open_start=False,
    use_open_end=False,
    output_dir=OUTPUT_DIR,
    cleanup_temp=True,
    rna=True,
)

print(f"\n{'='*60}")
print(f"Pipeline complete!")
print(f"  f5c version:      {metadata.f5c_version}")
print(f"  Contigs total:    {metadata.n_contigs_total}")
print(f"  Contigs passed:   {metadata.n_contigs_passed_filter}")
print(f"  Contigs skipped:  {metadata.n_contigs_skipped}")
print(f"  use_cuda:         {metadata.use_cuda}")
print(f"  Results saved to: {OUTPUT_DIR / 'pipeline_results.pkl'}")

## 2.5 Known Modification Sites (E. coli 16S & 23S rRNA)

Define the set of experimentally validated modification sites so we can
highlight them in downstream analyses. Positions are **1-based** to match
the eventalign output.

In [ ]:
# Known E. coli rRNA modifications
# Format: {(contig, position): (short_name, full_name)}
KNOWN_MODIFICATIONS = {
    # --- Pseudouridine (Psi) ---
    ("ecoli16S", 516):  ("Psi", "pseudouridine"),
    ("ecoli23S", 746):  ("Psi", "pseudouridine"),
    ("ecoli23S", 955):  ("Psi", "pseudouridine"),
    ("ecoli23S", 1911): ("Psi", "pseudouridine"),
    ("ecoli23S", 1917): ("Psi", "pseudouridine"),
    ("ecoli23S", 2457): ("Psi", "pseudouridine"),
    ("ecoli23S", 2504): ("Psi", "pseudouridine"),
    ("ecoli23S", 2580): ("Psi", "pseudouridine"),
    ("ecoli23S", 2604): ("Psi", "pseudouridine"),
    ("ecoli23S", 2605): ("Psi", "pseudouridine"),
    # --- N2-methylguanosine (m2G) ---
    ("ecoli16S", 966):  ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1207): ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1516): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 1835): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 2445): ("m2G", "N2-methylguanosine"),
    # --- 5-methylcytidine (m5C) ---
    ("ecoli16S", 967):  ("m5C", "5-methylcytidine"),
    ("ecoli16S", 1407): ("m5C", "5-methylcytidine"),
    ("ecoli23S", 1962): ("m5C", "5-methylcytidine"),
    # --- 5-methyluridine (m5U) ---
    ("ecoli23S", 747):  ("m5U", "5-methyluridine"),
    ("ecoli23S", 1939): ("m5U", "5-methyluridine"),
    # --- N6,N6-dimethyladenosine (m66A) ---
    ("ecoli16S", 1518): ("m66A", "N6,N6-dimethyladenosine"),
    ("ecoli16S", 1519): ("m66A", "N6,N6-dimethyladenosine"),
    # --- N6-methyladenosine (m6A) ---
    ("ecoli23S", 1618): ("m6A", "N6-methyladenosine"),
    ("ecoli23S", 2030): ("m6A", "N6-methyladenosine"),
    # --- N7-methylguanosine (m7G) ---
    ("ecoli16S", 527):  ("m7G", "N7-methylguanosine"),
    ("ecoli23S", 2069): ("m7G", "N7-methylguanosine"),
    # --- Single-occurrence modifications ---
    ("ecoli23S", 2498): ("Cm",    "2\u2032-O-methyl cytosine"),
    ("ecoli23S", 2449): ("D",     "dihydrouridine"),
    ("ecoli23S", 2251): ("Gm",    "2\u2032-O-methyl guanine"),
    ("ecoli23S", 2552): ("Um",    "2\u2032-O-methyl uridine"),
    ("ecoli23S", 2501): ("ho5C",  "5-hydroxycytidine"),
    ("ecoli23S", 745):  ("m1G",   "1-methylguanosine"),
    ("ecoli23S", 2503): ("m2A",   "2-methyladenosine"),
    ("ecoli23S", 1915): ("m3Psi", "3-methylpseudouridine"),
    ("ecoli16S", 1498): ("m3U",   "3-methyluridine"),
    ("ecoli16S", 1402): ("m4Cm",  "N4,2\u2032-O-dimethylcytidine"),
}

# Build a quick-lookup set
KNOWN_MOD_SET = set(KNOWN_MODIFICATIONS.keys())

# Summary table
from collections import Counter
mod_counts = Counter(v[0] for v in KNOWN_MODIFICATIONS.values())
print(f"Total known modification sites: {len(KNOWN_MODIFICATIONS)}")
print(f"Modification types: {len(mod_counts)}\n")
print(f"{'Type':<8s} {'Count':>5s}  {'Full name'}")
print(f"{'----':<8s} {'-----':>5s}  {'---------'}")
for mod_type, count in mod_counts.most_common():
    full = next(v[1] for v in KNOWN_MODIFICATIONS.values() if v[0] == mod_type)
    print(f"{mod_type:<8s} {count:5d}  {full}")

## 3. Inspect Results

Explore the pipeline output: per-contig summaries, number of positions with
DTW matrices, and matrix dimensions.

In [ ]:
# If you want to reload from disk instead of re-running:
# results, metadata = load_results(OUTPUT_DIR / "pipeline_results.pkl")

print(f"Contigs with results: {len(results)}\n")

for contig_name, contig_result in sorted(results.items()):
    n_pos = len(contig_result.positions)
    print(f"Contig: {contig_name}")
    print(f"  Native depth: {contig_result.native_depth:.1f}")
    print(f"  IVT depth:    {contig_result.ivt_depth:.1f}")
    print(f"  Positions with DTW matrices: {n_pos}")
    if n_pos > 0:
        # Show first few positions
        for pos, pr in list(contig_result.positions.items())[:5]:
            print(
                f"    pos={pos:6d}  kmer={pr.reference_kmer:6s}  "
                f"native_reads={pr.n_native_reads}  ivt_reads={pr.n_ivt_reads}  "
                f"matrix={pr.distance_matrix.shape}"
            )
        if n_pos > 5:
            print(f"    ... and {n_pos - 5} more positions")
    print()

## 4. Summary Statistics

In [ ]:
total_positions = 0
total_native_reads = 0
total_ivt_reads = 0
all_distances = []

for contig_result in results.values():
    for pr in contig_result.positions.values():
        total_positions += 1
        total_native_reads += pr.n_native_reads
        total_ivt_reads += pr.n_ivt_reads
        # Upper triangle (unique pairs)
        triu = pr.distance_matrix[np.triu_indices_from(pr.distance_matrix, k=1)]
        all_distances.extend(triu.tolist())

all_distances = np.array(all_distances)

print(f"Total positions with DTW results: {total_positions}")
print(f"Total native reads across positions: {total_native_reads}")
print(f"Total IVT reads across positions: {total_ivt_reads}")
print(f"Total unique pairwise distances: {len(all_distances)}")
if len(all_distances) > 0:
    print(f"\nDistance statistics:")
    print(f"  min:    {all_distances.min():.4f}")
    print(f"  max:    {all_distances.max():.4f}")
    print(f"  mean:   {all_distances.mean():.4f}")
    print(f"  median: {np.median(all_distances):.4f}")
    print(f"  std:    {all_distances.std():.4f}")

## 5. Visualize DTW Distance Matrices

For each contig, show heatmaps of the DTW distance matrices at selected positions.
The matrix rows/columns are ordered: **native reads first, then IVT reads**.
The dividing line between native and IVT is marked.

In [ ]:
def plot_dtw_matrix(pr, contig_name, ax=None):
    """Plot a single DTW distance matrix as a heatmap."""
    if ax is None:
        _, ax = plt.subplots(1, 1, figsize=(6, 5))

    mat = pr.distance_matrix
    n_nat = pr.n_native_reads
    n_ivt = pr.n_ivt_reads
    n_total = n_nat + n_ivt

    im = ax.imshow(mat, cmap="viridis", interpolation="nearest")
    plt.colorbar(im, ax=ax, shrink=0.8, label="DTW distance")

    # Draw boundary between native and IVT
    if n_nat > 0 and n_ivt > 0:
        ax.axhline(n_nat - 0.5, color="red", linewidth=1.5, linestyle="--")
        ax.axvline(n_nat - 0.5, color="red", linewidth=1.5, linestyle="--")

    ax.set_title(
        f"{contig_name}\npos={pr.position} kmer={pr.reference_kmer}\n"
        f"native={n_nat}, ivt={n_ivt}",
        fontsize=10,
    )
    ax.set_xlabel("Read index")
    ax.set_ylabel("Read index")

    # Label native vs IVT regions
    if n_total <= 30:
        labels = (
            [f"N{i}" for i in range(n_nat)]
            + [f"I{i}" for i in range(n_ivt)]
        )
        ax.set_xticks(range(n_total))
        ax.set_xticklabels(labels, fontsize=6, rotation=90)
        ax.set_yticks(range(n_total))
        ax.set_yticklabels(labels, fontsize=6)

    return ax

In [ ]:
MAX_POSITIONS_PER_CONTIG = 6  # how many positions to plot per contig

for contig_name, contig_result in sorted(results.items()):
    positions = contig_result.positions
    if not positions:
        print(f"Skipping {contig_name}: no positions with DTW results")
        continue

    # Prioritize known modification positions, then fill with evenly spaced others
    sorted_positions = sorted(positions.keys())
    known_pos = [p for p in sorted_positions if (contig_name, p) in KNOWN_MOD_SET]
    other_pos = [p for p in sorted_positions if (contig_name, p) not in KNOWN_MOD_SET]

    selected = known_pos[:MAX_POSITIONS_PER_CONTIG]
    remaining_slots = MAX_POSITIONS_PER_CONTIG - len(selected)
    if remaining_slots > 0 and other_pos:
        indices = np.linspace(0, len(other_pos) - 1, remaining_slots, dtype=int)
        selected += [other_pos[i] for i in indices]
    selected = sorted(selected)
    n_show = len(selected)

    n_cols = min(3, n_show)
    n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 4.5 * n_rows),
        squeeze=False,
    )
    fig.suptitle(f"Contig: {contig_name}", fontsize=13, fontweight="bold")

    for idx, pos in enumerate(selected):
        row, col = divmod(idx, n_cols)
        ax = plot_dtw_matrix(positions[pos], contig_name, ax=axes[row][col])
        # Mark known modifications with a star in the title
        if (contig_name, pos) in KNOWN_MOD_SET:
            mod_name = KNOWN_MODIFICATIONS[(contig_name, pos)][0]
            ax.set_title(ax.get_title() + f"\n\u2605 {mod_name}", color="red", fontsize=10)

    # Hide unused axes
    for idx in range(n_show, n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row][col].set_visible(False)

    plt.tight_layout()
    plt.show()

### 5b. Boxplot: Within-group vs Between-group Distances

For the same selected positions, show side-by-side boxplots of the three
distance categories: **IVT–IVT** (within), **Native–Native** (within),
and **IVT–Native** (between). If between-group distances are systematically
larger, the position likely carries a modification.

In [ ]:
def _decompose_distances(pr):
    """Split distance matrix into within-native, within-IVT, and between distances."""
    mat = pr.distance_matrix
    n_nat = pr.n_native_reads
    nn_block = mat[:n_nat, :n_nat]
    nn = nn_block[np.triu_indices_from(nn_block, k=1)]
    ii_block = mat[n_nat:, n_nat:]
    ii = ii_block[np.triu_indices_from(ii_block, k=1)]
    ni = mat[:n_nat, n_nat:].ravel()
    return nn, ii, ni


for contig_name, contig_result in sorted(results.items()):
    positions = contig_result.positions
    if not positions:
        continue

    # Reuse the same selection logic as the heatmap cell
    sorted_positions = sorted(positions.keys())
    known_pos = [p for p in sorted_positions if (contig_name, p) in KNOWN_MOD_SET]
    other_pos = [p for p in sorted_positions if (contig_name, p) not in KNOWN_MOD_SET]

    selected = known_pos[:MAX_POSITIONS_PER_CONTIG]
    remaining_slots = MAX_POSITIONS_PER_CONTIG - len(selected)
    if remaining_slots > 0 and other_pos:
        indices = np.linspace(0, len(other_pos) - 1, remaining_slots, dtype=int)
        selected += [other_pos[i] for i in indices]
    selected = sorted(selected)
    n_show = len(selected)

    n_cols = min(3, n_show)
    n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(5 * n_cols, 4 * n_rows),
        squeeze=False,
    )
    fig.suptitle(
        f"Contig: {contig_name} — Distance Boxplots",
        fontsize=13, fontweight="bold",
    )

    box_colors = ["#4C72B0", "#55A868", "#C44E52"]  # blue, green, red

    for idx, pos in enumerate(selected):
        row, col = divmod(idx, n_cols)
        ax = axes[row][col]
        pr = positions[pos]
        nn, ii, ni = _decompose_distances(pr)

        data = []
        labels = []
        colors_used = []
        if len(ii) > 0:
            data.append(ii)
            labels.append(f"IVT–IVT\n(n={len(ii)})")
            colors_used.append(box_colors[1])
        if len(nn) > 0:
            data.append(nn)
            labels.append(f"Nat–Nat\n(n={len(nn)})")
            colors_used.append(box_colors[0])
        if len(ni) > 0:
            data.append(ni)
            labels.append(f"IVT–Nat\n(n={len(ni)})")
            colors_used.append(box_colors[2])

        if not data:
            ax.text(0.5, 0.5, "No pairs", ha="center", va="center", transform=ax.transAxes)
            continue

        bp = ax.boxplot(
            data,
            labels=labels,
            patch_artist=True,
            widths=0.6,
            showfliers=True,
            flierprops=dict(marker=".", markersize=3, alpha=0.4),
        )
        for patch, color in zip(bp["boxes"], colors_used):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)

        title = f"pos={pos}  kmer={pr.reference_kmer}"
        if (contig_name, pos) in KNOWN_MOD_SET:
            mod_name = KNOWN_MODIFICATIONS[(contig_name, pos)][0]
            title += f"\n\u2605 {mod_name}"
            ax.set_title(title, fontsize=9, color="red")
        else:
            ax.set_title(title, fontsize=9)

        ax.set_ylabel("DTW distance")
        ax.grid(True, axis="y", alpha=0.3)

    # Hide unused axes
    for idx in range(n_show, n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row][col].set_visible(False)

    plt.tight_layout()
    plt.show()

## 6. Native vs IVT Distance Comparison

For each position, compare within-group distances (native-native, IVT-IVT)
against between-group distances (native-IVT). Modified positions should show
larger between-group distances.

In [ ]:
def decompose_distances(pr):
    """Split a distance matrix into within-native, within-IVT, and between-group distances."""
    mat = pr.distance_matrix
    n_nat = pr.n_native_reads
    n_ivt = pr.n_ivt_reads

    # Native-Native (upper triangle of top-left block)
    nn_block = mat[:n_nat, :n_nat]
    nn = nn_block[np.triu_indices_from(nn_block, k=1)]

    # IVT-IVT (upper triangle of bottom-right block)
    ii_block = mat[n_nat:, n_nat:]
    ii = ii_block[np.triu_indices_from(ii_block, k=1)]

    # Native-IVT (full off-diagonal block)
    ni = mat[:n_nat, n_nat:].ravel()

    return nn, ii, ni

In [ ]:
rows = []
for contig_name, contig_result in results.items():
    for pos, pr in contig_result.positions.items():
        nn, ii, ni = decompose_distances(pr)
        rows.append({
            "contig": contig_name,
            "position": pos,
            "kmer": pr.reference_kmer,
            "n_native": pr.n_native_reads,
            "n_ivt": pr.n_ivt_reads,
            "mean_native_native": float(nn.mean()) if len(nn) > 0 else np.nan,
            "mean_ivt_ivt": float(ii.mean()) if len(ii) > 0 else np.nan,
            "mean_native_ivt": float(ni.mean()) if len(ni) > 0 else np.nan,
        })

if rows:
    import pandas as pd
    df = pd.DataFrame(rows)
    df["ratio_between_within"] = df["mean_native_ivt"] / (
        (df["mean_native_native"] + df["mean_ivt_ivt"]) / 2
    )

    # Annotate known modification sites
    df["modification"] = df.apply(
        lambda r: KNOWN_MODIFICATIONS.get((r["contig"], r["position"]), ("", ""))[0],
        axis=1,
    )
    df["is_modified"] = df["modification"] != ""

    df = df.sort_values("ratio_between_within", ascending=False)

    n_mod_found = df["is_modified"].sum()
    print(f"Positions analyzed: {len(df)}")
    print(f"Known modification sites found in results: {n_mod_found} / {len(KNOWN_MODIFICATIONS)}")
    print()

    # Show known-modification rows first, then the rest
    df_mod = df[df["is_modified"]].copy()
    df_unmod = df[~df["is_modified"]].copy()
    if len(df_mod) > 0:
        print("=== Known modification sites (sorted by ratio) ===")
        print(df_mod.to_string(index=False, float_format="%.3f"))
        print()
    print(f"=== All positions (top 50 by ratio) ===")
    print(df.head(50).to_string(index=False, float_format="%.3f"))
else:
    print("No positions with results to analyze.")

In [ ]:
if rows:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: Distribution of mean distances by group
    ax = axes[0]
    ax.hist(df["mean_native_native"].dropna(), bins=30, alpha=0.6, label="Native-Native", color="tab:blue")
    ax.hist(df["mean_ivt_ivt"].dropna(), bins=30, alpha=0.6, label="IVT-IVT", color="tab:green")
    ax.hist(df["mean_native_ivt"].dropna(), bins=30, alpha=0.6, label="Native-IVT", color="tab:red")
    ax.set_xlabel("Mean DTW distance")
    ax.set_ylabel("Count (positions)")
    ax.set_title("Distribution of mean pairwise distances")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Right: Between/within ratio
    ax = axes[1]
    ratios = df["ratio_between_within"].dropna()
    ax.hist(ratios, bins=30, color="tab:purple", alpha=0.7)
    ax.axvline(1.0, color="gray", linestyle="--", linewidth=1.5, label="ratio=1.0")
    if len(ratios) > 0:
        ax.axvline(ratios.median(), color="red", linestyle="-", linewidth=1.5, label=f"median={ratios.median():.2f}")
    ax.set_xlabel("Between-group / Within-group distance ratio")
    ax.set_ylabel("Count (positions)")
    ax.set_title("Native-IVT separation")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 7. Per-Position Scatter: Within vs Between

Each point is a position. Positions above the diagonal have larger between-group
distance (potential modification sites).

In [ ]:
if rows:
    fig, ax = plt.subplots(figsize=(8, 8))

    within = (df["mean_native_native"] + df["mean_ivt_ivt"]) / 2
    between = df["mean_native_ivt"]
    is_mod = df["is_modified"]

    # Unmodified positions (grey, background)
    ax.scatter(
        within[~is_mod], between[~is_mod],
        alpha=0.3, s=12, c="tab:gray", label="unmodified", zorder=2,
    )

    # Known modification positions (colored by type, on top)
    mod_types_in_data = df.loc[is_mod, "modification"].unique()
    cmap = plt.cm.get_cmap("tab10", max(len(mod_types_in_data), 1))
    for idx, mt in enumerate(sorted(mod_types_in_data)):
        mask = df["modification"] == mt
        ax.scatter(
            within[mask], between[mask],
            s=60, marker="^", edgecolors="black", linewidths=0.5,
            c=[cmap(idx)], label=mt, zorder=5, alpha=0.9,
        )
        # Annotate with contig:position
        for _, row in df[mask].iterrows():
            w = (row["mean_native_native"] + row["mean_ivt_ivt"]) / 2
            b = row["mean_native_ivt"]
            ax.annotate(
                f"{row['contig']}:{row['position']}",
                (w, b), fontsize=6, alpha=0.8,
                xytext=(4, 4), textcoords="offset points",
            )

    # Diagonal reference line
    lims = [
        min(within.min(), between.min()) * 0.95,
        max(within.max(), between.max()) * 1.05,
    ]
    ax.plot(lims, lims, "--", color="gray", linewidth=1, label="y=x")

    ax.set_xlabel("Mean within-group DTW distance")
    ax.set_ylabel("Mean between-group DTW distance (native vs IVT)")
    ax.set_title("Within vs Between-group distances per position\n"
                 "(\u25b2 = known modification sites)")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal")

    plt.tight_layout()
    plt.show()

## 8. Modification Probability Estimation

Run three algorithms on each position's DTW distance matrix to estimate
per-read modification probabilities:

1. **Distance-to-IVT** — median distance to IVT baseline, Normal EM calibration
2. **kNN IVT-Purity** — weighted IVT fraction among k nearest neighbors, Beta EM calibration
3. **MDS + GMM** — classical MDS embedding + IVT-anchored Gaussian mixture

In [ ]:
from baleen.eventalign import (
    AlgorithmName,
    compute_modification_probabilities,
)

all_probs: dict[str, dict[int, dict[AlgorithmName, object]]] = {}

for contig_name, contig_result in results.items():
    all_probs[contig_name] = {}
    for pos, pr in contig_result.positions.items():
        mp = compute_modification_probabilities(
            pr.distance_matrix,
            n_native=pr.n_native_reads,
            n_ivt=pr.n_ivt_reads,
            position=pos,
        )
        all_probs[contig_name][pos] = mp

n_positions = sum(len(v) for v in all_probs.values())
print(f"Computed modification probabilities for {n_positions} positions")

### 8a. Per-Read Probability Distributions (Boxplots)

For each selected position, compare modification probability distributions
for native (blue) and IVT (green) reads across all three algorithms.

In [ ]:
ALG_LABELS = {
    AlgorithmName.DISTANCE_TO_IVT: "Dist-to-IVT",
    AlgorithmName.KNN_IVT_PURITY: "kNN Purity",
    AlgorithmName.MDS_GMM: "MDS+GMM",
}
ALG_ORDER = [AlgorithmName.DISTANCE_TO_IVT, AlgorithmName.KNN_IVT_PURITY, AlgorithmName.MDS_GMM]

for contig_name in sorted(all_probs.keys()):
    contig_positions = all_probs[contig_name]
    sorted_pos = sorted(contig_positions.keys())

    known_pos = [p for p in sorted_pos if (contig_name, p) in KNOWN_MOD_SET]
    other_pos = [p for p in sorted_pos if (contig_name, p) not in KNOWN_MOD_SET]
    selected = known_pos[:MAX_POSITIONS_PER_CONTIG]
    remaining = MAX_POSITIONS_PER_CONTIG - len(selected)
    if remaining > 0 and other_pos:
        idx = np.linspace(0, len(other_pos) - 1, remaining, dtype=int)
        selected += [other_pos[i] for i in idx]
    selected = sorted(selected)
    if not selected:
        continue

    fig, axes = plt.subplots(
        len(selected), 3,
        figsize=(14, 3.5 * len(selected)),
        squeeze=False,
    )
    fig.suptitle(f"Modification Probabilities — {contig_name}",
                 fontsize=14, fontweight="bold", y=1.01)

    for row_idx, pos in enumerate(selected):
        mp_dict = contig_positions[pos]
        for col_idx, alg in enumerate(ALG_ORDER):
            ax = axes[row_idx][col_idx]
            mp = mp_dict[alg]

            if mp.null_gate_active:
                ax.text(0.5, 0.5, "Null gate\n(no modification signal)",
                        ha="center", va="center", transform=ax.transAxes,
                        fontsize=10, color="gray")
                ax.set_ylim(0, 1)
            else:
                data = [mp.native_probabilities, mp.ivt_probabilities]
                bp = ax.boxplot(data, labels=["Native", "IVT"],
                               patch_artist=True, widths=0.5)
                bp["boxes"][0].set_facecolor("#5B9BD5")
                bp["boxes"][1].set_facecolor("#70AD47")
                ax.set_ylim(-0.05, 1.05)

            is_known = (contig_name, pos) in KNOWN_MOD_SET
            mod_label = ""
            if is_known:
                mod_label = f" \u2605 {KNOWN_MODIFICATIONS[(contig_name, pos)][0]}"
            title = f"pos {pos}{mod_label}"
            if row_idx == 0:
                title = f"{ALG_LABELS[alg]}\n{title}"
            ax.set_title(title, fontsize=9,
                         color="red" if is_known else "black")
            if col_idx == 0:
                ax.set_ylabel("P(modified)")

    plt.tight_layout()
    plt.show()

### 8b. Probability Heatmaps

Heatmap view showing modification probability for every read at selected positions,
ordered by algorithm.  Rows = reads (native first, then IVT, separated by a red line).

In [ ]:
for contig_name in sorted(all_probs.keys()):
    contig_positions = all_probs[contig_name]
    sorted_pos = sorted(contig_positions.keys())

    known_pos = [p for p in sorted_pos if (contig_name, p) in KNOWN_MOD_SET]
    other_pos = [p for p in sorted_pos if (contig_name, p) not in KNOWN_MOD_SET]
    selected = known_pos[:MAX_POSITIONS_PER_CONTIG]
    remaining = MAX_POSITIONS_PER_CONTIG - len(selected)
    if remaining > 0 and other_pos:
        idx = np.linspace(0, len(other_pos) - 1, remaining, dtype=int)
        selected += [other_pos[i] for i in idx]
    selected = sorted(selected)
    if not selected:
        continue

    fig, axes = plt.subplots(
        1, len(selected),
        figsize=(3 * len(selected), 8),
        squeeze=False,
    )
    fig.suptitle(f"Probability Heatmaps — {contig_name} (Dist-to-IVT)",
                 fontsize=13, fontweight="bold")

    for col_idx, pos in enumerate(selected):
        ax = axes[0][col_idx]
        mp = contig_positions[pos][AlgorithmName.DISTANCE_TO_IVT]
        probs = mp.probabilities.reshape(-1, 1)

        im = ax.imshow(probs, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
        ax.axhline(mp.n_native - 0.5, color="red", linewidth=1.5, linestyle="--")

        ax.set_xticks([])
        ax.set_ylabel("Read index" if col_idx == 0 else "")

        is_known = (contig_name, pos) in KNOWN_MOD_SET
        mod_label = ""
        if is_known:
            mod_label = f"\n\u2605 {KNOWN_MODIFICATIONS[(contig_name, pos)][0]}"
        ax.set_title(f"pos {pos}{mod_label}", fontsize=9,
                     color="red" if is_known else "black")

    fig.colorbar(im, ax=axes[0].tolist(), label="P(modified)", shrink=0.6)
    plt.tight_layout()
    plt.show()

### 8c. Algorithm Agreement

Compare mean native probability from each algorithm pair.  Points on the
diagonal indicate perfect agreement; known modification sites are highlighted.

In [ ]:
import itertools

pairs = list(itertools.combinations(ALG_ORDER, 2))

fig, axes = plt.subplots(1, len(pairs), figsize=(6 * len(pairs), 5), squeeze=False)
fig.suptitle("Algorithm Agreement — Mean Native P(modified)",
             fontsize=14, fontweight="bold")

for ax_idx, (alg_a, alg_b) in enumerate(pairs):
    ax = axes[0][ax_idx]
    xs_known, ys_known = [], []
    xs_other, ys_other = [], []

    for contig_name, contig_positions in all_probs.items():
        for pos, mp_dict in contig_positions.items():
            ma, mb = mp_dict[alg_a], mp_dict[alg_b]
            x = float(np.mean(ma.native_probabilities))
            y = float(np.mean(mb.native_probabilities))
            if (contig_name, pos) in KNOWN_MOD_SET:
                xs_known.append(x)
                ys_known.append(y)
            else:
                xs_other.append(x)
                ys_other.append(y)

    ax.scatter(xs_other, ys_other, alpha=0.4, s=20, color="steelblue", label="other")
    ax.scatter(xs_known, ys_known, alpha=0.9, s=50, color="red", marker="^", label="known mod")
    ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
    ax.set_xlabel(ALG_LABELS[alg_a])
    ax.set_ylabel(ALG_LABELS[alg_b])
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect("equal")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 8d. Summary Table

For each position show the mean native probability per algorithm, the
null-gate status, and whether it is a known modification site.

In [ ]:
import pandas as pd

summary_rows = []
for contig_name, contig_positions in all_probs.items():
    for pos, mp_dict in sorted(contig_positions.items()):
        row = {"contig": contig_name, "position": pos}
        is_known = (contig_name, pos) in KNOWN_MOD_SET
        row["known_mod"] = KNOWN_MODIFICATIONS[(contig_name, pos)][0] if is_known else ""
        for alg in ALG_ORDER:
            mp = mp_dict[alg]
            label = ALG_LABELS[alg]
            row[f"{label}_mean_native"] = round(float(np.mean(mp.native_probabilities)), 4)
            row[f"{label}_mean_ivt"] = round(float(np.mean(mp.ivt_probabilities)), 4)
            row[f"{label}_null_gate"] = mp.null_gate_active
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print(f"{len(df_summary)} positions total")
df_summary.head(20)

---

**Done.** Results are saved at the output directory and can be reloaded with:

```python
from baleen.eventalign import load_results
results, metadata = load_results("output/eventalign_test/pipeline_results.pkl")
```